# 🎯 Smart Resume Keyword Matcher

> **Portfolio Project** | Classical NLP + TF-IDF + Cosine Similarity
>
> Compares a Resume PDF against a Job Description and returns:
> - An overall **match score (%)**
> - A **section breakdown** (Skills / Experience / Education)
> - A prioritized list of **missing keywords**
> - A **history log** of all past comparisons

---

**Tech Stack:** `pdfplumber` · `spaCy (en_core_web_sm)` · `scikit-learn (TF-IDF)` · `Gradio`

**Setup:**
```bash
pip install -r requirements.txt
python -m spacy download en_core_web_sm
```

## ⚙️ Setup & Imports

In [ ]:
# ── Setup Verification ────────────────────────────────────────────────────────
# Run this cell FIRST to verify all dependencies are installed correctly.
# If any import fails, run: pip install -r requirements.txt
# Then run: python -m spacy download en_core_web_sm

import sys
import importlib
import os

# Check all required packages are importable
REQUIRED = ["pdfplumber", "spacy", "sklearn", "gradio", "pandas", "numpy"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"❌ Missing packages: {missing}")
    print("   Fix: pip install -r requirements.txt")
else:
    print("✅ All dependencies found. You're offline-ready.")

# Load spaCy model — works fully offline after the first download
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
    print("✅ spaCy model 'en_core_web_sm' loaded.")
except OSError:
    print("❌ spaCy model not found.")
    print("   Fix: python -m spacy download en_core_web_sm")

# Verify history.json exists — create it if missing
HISTORY_PATH = "history.json"
if not os.path.exists(HISTORY_PATH):
    import json
    with open(HISTORY_PATH, "w") as f:
        json.dump([], f)
    print(f"✅ Created {HISTORY_PATH}")
else:
    print(f"✅ {HISTORY_PATH} found.")

print("\n🚀 Setup complete — ready to match resumes!")

## 📄 Phase 1: PDF Parsing & Section Detection

These two functions form the **input pipeline**:
1. `extract_text_from_pdf()` — reads a PDF and returns raw text
2. `detect_sections()` — splits resume text into Skills / Experience / Education buckets

Both functions are **offline** and require no internet at runtime.

In [ ]:
# ── PDF Text Extractor ────────────────────────────────────────────────────────
# Uses pdfplumber to extract raw text from every page of a resume PDF.
# pdfplumber handles multi-column layouts better than PyPDF2 — which matters
# because many modern resume templates use a two-column format.

import pdfplumber
import re

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract all text from a PDF file, page by page.

    Args:
        pdf_path: Absolute or relative path to the PDF file.

    Returns:
        A single cleaned string of all text from the PDF.
        Returns empty string if extraction fails (logs error to stdout).
    """
    full_text = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                page_text = page.extract_text()

                # Some pages may return None — e.g., image-only scanned pages
                if page_text:
                    full_text.append(page_text)
                else:
                    print(f"  ⚠️ Page {page_num}: No text extracted (may be image-based)")

    except FileNotFoundError:
        # Don't crash — return empty string and let the caller handle it gracefully
        print(f"❌ File not found: {pdf_path}")
        return ""
    except Exception as e:
        print(f"❌ Error reading PDF: {e}")
        return ""

    # Join all pages, then clean up messy whitespace
    combined = "\n".join(full_text)
    combined = re.sub(r'\n{3,}', '\n\n', combined)   # max 2 consecutive blank lines
    combined = re.sub(r'[ \t]{2,}', ' ', combined)   # collapse inline multiple spaces

    return combined.strip()


# ── Quick Smoke Test ──────────────────────────────────────────────────────────
# Place a PDF in sample_resumes/ and uncomment these lines to test:
# sample_text = extract_text_from_pdf("sample_resumes/test_resume.pdf")
# print(f"Extracted {len(sample_text)} characters")
# print(sample_text[:500])  # Preview first 500 chars

In [ ]:
# ── Section Detector ─────────────────────────────────────────────────────────
# Heuristically splits a resume's raw text into three sections:
# Skills, Experience, and Education.
#
# Strategy: scan line by line for section-header keywords.
# This is fast, offline, and 'good enough' for classical NLP scoring.
# Imperfect splits are smoothed out by cosine similarity averaging in Phase 2.

def detect_sections(text: str) -> dict:
    """
    Split resume text into Skills, Experience, Education, and General sections.

    Uses header-keyword detection to find section boundaries. Text between
    boundaries is assigned to that section. Unclassified text goes to 'general'.

    Args:
        text: Raw resume text (output of extract_text_from_pdf).

    Returns:
        dict with keys: 'skills', 'experience', 'education', 'general'
        Each value is a string of text belonging to that section.
    """
    if not text:
        # Handle empty input gracefully — return empty sections
        return {"skills": "", "experience": "", "education": "", "general": ""}

    # Keywords that signal the start of each section (case-insensitive)
    SECTION_KEYWORDS = {
        "skills": [
            "skill", "technical skill", "core competenc", "technologies",
            "tools", "programming", "languages", "proficienc"
        ],
        "experience": [
            "experience", "work history", "employment", "career",
            "professional background", "work experience", "internship"
        ],
        "education": [
            "education", "academic", "qualification", "degree",
            "university", "college", "schooling", "certification"
        ],
    }

    sections = {"skills": [], "experience": [], "education": [], "general": []}
    current_section = "general"  # Default — text before any known header

    for line in text.split("\n"):
        line_lower = line.lower().strip()

        # Only check for section change if this line is short enough to be a header.
        # The 40-char guard prevents long sentences that happen to contain keywords
        # (e.g., "5 years of Python experience") from being misclassified as headers.
        matched_section = None
        if len(line_lower) <= 40:
            for section, keywords in SECTION_KEYWORDS.items():
                if any(kw in line_lower for kw in keywords):
                    matched_section = section
                    break

        if matched_section:
            current_section = matched_section
        else:
            sections[current_section].append(line)

    # Join the accumulated lines back into strings
    return {k: "\n".join(v).strip() for k, v in sections.items()}


# ── Quick Smoke Test ──────────────────────────────────────────────────────────
# sample_sections = detect_sections(sample_text)
# for section, content in sample_sections.items():
#     preview = content[:150] if content else "(empty)"
#     print(f"\n── {section.upper()} ──\n{preview}")

## 🧠 Phase 2: NLP Scoring

*(Implementation in Phase 2 — Plan 2.x)*

Will implement:
- `preprocess_text()` — spaCy lemmatization & stopword removal
- `compute_match_score()` — TF-IDF vectorization + cosine similarity
- `find_missing_keywords()` — JD terms absent from resume

## 💾 Phase 3: History & Persistence

*(Implementation in Phase 3 — Plan 3.x)*

Will implement:
- `save_result()` — append comparison result to `history.json`
- `load_history()` — read all past comparisons

## 🖥️ Phase 4: Gradio UI

*(Implementation in Phase 4 — Plan 4.x)*

Will implement:
- PDF upload + JD text area + Analyze button
- Output panel: score gauge, section breakdown table, missing keywords
- History tab: past comparisons from `history.json`